In [3]:
import pandas as pd
import sqlite3

# Connect to your SQLite database
conn = sqlite3.connect("gaming_data.db")

# Query 1: Regional Leaderboards using DENSE_RANK()
query_leaderboard = """
SELECT 
    p.player_id,
    p.username,
    p.region,
    SUM(m.mmr_change) AS total_mmr,
    DENSE_RANK() OVER (PARTITION BY p.region ORDER BY SUM(m.mmr_change) DESC) AS regional_rank
FROM players p
JOIN matches m ON p.player_id = m.player_id
GROUP BY p.player_id, p.username, p.region
LIMIT 10;
"""

df_leaderboard = pd.read_sql_query(query_leaderboard, conn)
print("--- REGIONAL LEADERBOARD (TOP 10) ---")
display(df_leaderboard)

# Query 2: Performance & Ping Impact using JOIN & Aggregations
query_ping_impact = """
SELECT 
    p.region,
    ROUND(AVG(s.ping_ms), 2) AS avg_ping_ms,
    COUNT(m.match_id) AS total_matches,
    ROUND(COUNT(CASE WHEN m.outcome = 'Win' THEN 1 END) * 100.0 / COUNT(m.match_id), 2) AS win_rate_percentage
FROM players p
JOIN matches m ON p.player_id = m.player_id
JOIN sessions s ON p.player_id = s.player_id
GROUP BY p.region;
"""

df_ping = pd.read_sql_query(query_ping_impact, conn)
print("\n--- REGIONAL PING VS WIN RATE ---")
display(df_ping)

conn.close()

ModuleNotFoundError: No module named 'pandas'